In [1]:
from jinja2 import optimizer
%load_ext autoreload
%autoreload 2


import genesis as gs
import logging
gs.init(logging_level=logging.WARNING, backend=gs.gpu)
from buffer import Buffer
from network import Network
from make_environment import Go2WalkingEnv
from reward import Rewards


[I 03/13/26 16:25:00.508 2077421] [shell.py:_shell_pop_print@25] Graphical python shell detected, using wrapped sys.stdout
2026-03-13 16:25:02.706 Python[93115:2077421] ApplePersistenceIgnoreState: Existing state will not be touched. New state will be written to /var/folders/3y/snyjd7qd59d_gxq1x0y5gywh0000gn/T/org.python.python.savedState


In [4]:
reward_fn = Rewards()
num_envs = 100
max_steps = 100

env = Go2WalkingEnv(
    num_envs=num_envs,
    device="mps",
    show_viewer=False,
    use_terrain=False,  # Set to True for complex terrain
    episode_length_s=20.0,
    reward_fn=reward_fn
)

[Genesis] [16:27:08] [WARNING] Viewer option 'n_rendered_envs' is deprecated and will be removed in future release. Please use 'rendered_envs_idx' instead.
[Genesis] [16:27:12] [WARNING] Neutral robot position (qpos0) exceeds joint limits.


In [5]:
env.set_commands(lin_vel_x=1.0, lin_vel_y=0.0, ang_vel_yaw=0.0)
policy = Network(
    num_outputs=env.num_actions,
    num_inputs=env.num_obs,
    gamma=0.99,
    lmbda=0.0,
    epsilon=0.1,
)
buffer = Buffer(
    num_envs=num_envs,
    obs_dim=env.num_obs,
    act_dim=env.num_actions,
    max_length=max_steps,
    device='mps'
)

In [6]:
import torch
policy.apply(lambda m: torch.nn.init.xavier_uniform_(m.weight) if hasattr(m, 'weight') else None)

Network(
  (shared): Sequential(
    (0): Linear(in_features=48, out_features=512, bias=True)
    (1): ELU(alpha=1.0)
    (2): Linear(in_features=512, out_features=256, bias=True)
    (3): ELU(alpha=1.0)
    (4): Linear(in_features=256, out_features=128, bias=True)
    (5): ELU(alpha=1.0)
  )
  (actor): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ELU(alpha=1.0)
    (2): Linear(in_features=64, out_features=32, bias=True)
  )
  (actor_mean): Linear(in_features=32, out_features=12, bias=True)
  (critic): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ELU(alpha=1.0)
    (2): Linear(in_features=64, out_features=1, bias=True)
  )
)

In [7]:
def adjust_scales(update):
    if update < 200:
        scales = {
            "tracking_lin_vel_x": 2.0,
            "tracking_ang_vel": 0.2,
            "x_progress": 1.0,
            "lin_vel_z": -0.1,
            "lin_vel_y": -0.05,
            "action_rate": -0.002,
            "similar_to_default": -0.005,
            "termination": -2.0,
            "sideway_movement": -0.05,
        }
        reward_fn.scales = scales
    elif update < 500:
        scales = {
            "tracking_lin_vel_x": 2.5,
            "tracking_ang_vel": 0.4,
            "x_progress": 0.5,
            "lin_vel_z": -0.15,
            "lin_vel_y": -0.1,
            "action_rate": -0.005,
            "similar_to_default": -0.01,
            "termination": -2.0,
            "sideway_movement": -0.1,
        }
    else:
        scales = {
            "tracking_lin_vel_x": 2.0,
            "tracking_ang_vel": 0.5,
            "x_progress": 0.2,
            "lin_vel_z": -0.2,
            "lin_vel_y": -0.1,
            "action_rate": -0.01,
            "similar_to_default": -0.02,
            "termination": -2.0,
            "sideway_movement": -0.1,
        }



In [ ]:
optim = torch.optim.Adam(policy.parameters(), lr=5e-4, eps=1e-8)
#torch.autograd.set_detect_anomaly(True)
num_updates = 1000
steps_per_update = 2048
update_epochs = 5
minibatch_size = 128
#max_length =
device ='mps'
buffer = Buffer(
    num_envs=num_envs,
    obs_dim=env.num_obs,
    act_dim=env.num_actions,
    max_length=steps_per_update,
    device='mps'
)

for i in range(num_updates):
    adjust_scales(i)
    print(f"Running Sim: i: {i}")
    with torch.no_grad():
        buffer.reset()
        obs = env.reset()
        buffer.init_obs(obs, policy.get_value(obs))
        for step in range(steps_per_update):
            obs = obs.to(device)
            actions, value = policy.get_actions(obs)
            log_probs, log_probs_value, entropy = policy.compute_log_probs(obs, actions)
            next_obs , reward, done, info = env.step(actions)
            buffer.add_step(next_obs, actions, log_probs, reward, done, value)

            obs = next_obs

            if done.any():
                obs = env.reset()

        buffer.compute_returns_and_advantages(gamma=0.99, lmbda=0.95)
    print(f"Running Epochs: i: {i}")
    for epoch in range(update_epochs):
        batch = buffer.get_batch(minibatch_size)

        log_probs_new, values_new, entropy = policy.compute_log_probs(batch['obs'], batch['actions'])
        #print(batch)
        critic_loss, actor_loss = policy.compute_loss(states= batch['obs'],
                                                      actions= batch['actions'],
                                                      advantages=batch['advantages'],
                                                      critic_targets=batch['values'],
                                                      log_probs_old=batch['log_probs'],
                                                      returns = batch['returns'],)
        entropy_loss = -0.01 * entropy.mean()

        loss = actor_loss + 0.5 * critic_loss + entropy_loss

        optim.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(policy.parameters(), 0.5) # gradient clipping
        optim.step()

        if i % 10 == 0:
            print(f"Update {i}: Loss={loss.item():.3f}, AvgRew={buffer.rewards.mean():.3f}")



Update 0: Loss=15.902, AvgRew=0.285
Update 0: Loss=215.929, AvgRew=0.285
Update 0: Loss=66.436, AvgRew=0.285
Update 0: Loss=13.538, AvgRew=0.285
Update 0: Loss=12.401, AvgRew=0.285


In [8]:
import os
import torch

def save_checkpoint(path, policy, optim, update, avg_rew, extra=None):
    checkpoint = {
        "update": update,
        "model_state_dict": policy.state_dict(),
        "optimizer_state_dict": optim.state_dict(),
        "avg_reward": float(avg_rew),
    }
    if extra is not None:
        checkpoint.update(extra)

    os.makedirs(os.path.dirname(path), exist_ok=True)
    torch.save(checkpoint, path)


In [9]:
save_checkpoint(
        path=f"/Users/felix/PycharmProjects/Genesis-Dog-Walking/test/go2_update_tmp.pt",
        policy=policy,
        optim=optim,
        update=100,
        avg_rew=buffer.rewards.mean().item(),
    )